# English to SQL Workshop: Netflix catalog and Gemini API

## Before we begin
1. Navigate to [aistudio.google.com](https://aistudio.google.com)
2. Click `Get Started` and sign-in with a Google account (WM email should work too!)
3. At the bottom left, you'll find a key icon. Click it, and now in the top right, click `Create API key`.
4. Name your key (i.e. "English2SQL") and select a project.
5. After the key is created, copy it to have it handy.
6. In this Colab notebook, you'll see a key icon on the left. Click it, and then click `Gemini API keys`. Choose `Import key from Google AI Studio`, and click import. You can also click `Add new secret`, name it `GEMINI_API_KEY`, and paste the key you copied from the previous step. **DO NOT SHARE YOUR KEY WITH ANYONE!**
7. Go to PLACEHOLDER to download the data we will be using. In this Colab notebook, you'll find a folder icon on the left. Click it, and upload `netflix.db` to this session.

That's it! We're ready to get started.

In [ ]:
%pip -q install google-genai

In [ ]:
from google import genai
from getpass import getpass
import pandas as pd
import sqlite3

# 1. Connecting to the API and sending a message

In [ ]:
client = genai.Client(api_key=getpass("Enter your Gemini API key:"))

In [ ]:
test = "PUT YOUR QUESTION HERE"

response = client.models.generate_content(model='gemini-3.5-flash-lite',
                                          contents=question)
print(response.text)

# 2. Connecting to the database and running a few queries

In [ ]:
connection = sqlite3.connect("file:/content/data.db?mode=ro",
                             uri=True)

In [ ]:
example_query = "SELECT * FROM sqlite_master;"
pd.read_sql_query(sql=example_query,
                  con=connection)


In [ ]:
example_query = "SELECT title, release_year, content_rating, duration_minutes FROM titles WHERE title_id='s42'"
pd.read_sql_query(sql=example_query,
                  con=connection)

## Let's go back to the first query we ran...
Context: `sqlite_master` is an internal table used by sqlite which represents its schema. It describes objects like tables, indices, views, and triggers which organize the database.

In [ ]:
schema_rows = pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table' ORDER BY name;", connection)

In [ ]:
schema = "\n\n".join(schema_rows['sql'])
print(schema)

## ...and now we have the creation instructions for our tables!
Idea: If we feed this schema to Gemini in our prompts, Gemini doesn't need to invent tables or query columns that don't exist. The model already has the context.

## Framing the right questions for Gemini
Let's inspect the tables a little bit more granularly

In [ ]:
example_query = "SELECT * FROM titles;"
pd.read_sql_query(sql = example_query,
                  con=connection)

In [ ]:
example_query = "SELECT * from title_genres;"
pd.read_sql_query(sql = example_query,
                  con=connection)

We want to inspect the data ourselves in case there are any anomalies or details we should note when building our prompt (see Rules below).

# 3. Talking with Gemini about the data
Based on the tables above, think of some questions that you might want to ask Gemini.

Examples:
1) *What TV shows released after 2020 have more than one season?*
2) *What PG movies are under 90 minutes long?*
3) *How many documentaries are in this dataset?*

First, we'll assemble our prompt!

In [ ]:
def assemble_prompt(schema: str, question: str) -> str:
  prompt= f"""
  Write a SQLite query that answers the question below.

  Question:
  {question}

  Schema:
  {schema}

  Rules:
  - Use only the tables and columns in the schema provided. Do not invent columns or table names.
  - Generate a single read-only SELECT query.
  - 'content_rating' is an age classification, not an audience score.
  - 'duration_minutes' applies to movies; seasons applies to TV shows.
  - Return only SQL. Do not include markdown formatting or explanations.
  """
  return prompt

## Example Question 1

In [ ]:
question = "What TV shows released after 2020 have more than one season?"

prompt = assemble_prompt(schema=schema, question=question)

In [ ]:
response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt)

generated_sql = response.text.strip()

print(generated_sql)

In [ ]:
# Let's see what these records are:
pd.read_sql_query(sql=generated_sql,
                  con=connection)

## Example Question 2

In [ ]:
question = "What PG movies are under 90 minutes long?"

prompt = assemble_prompt(schema=schema, question=question)

In [ ]:
response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt)

generated_sql = response.text.strip()

print(generated_sql)

In [ ]:
# Let's see what these records are:
pd.read_sql_query(sql=generated_sql,
                  con=connection)

## Example Question 3

In [ ]:
question = "How many documentaries are in this dataset?"

prompt = assemble_prompt(schema=schema, question=question)

In [ ]:
response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt)

generated_sql = response.text.strip()

print(generated_sql)

In [ ]:
# Let's see what these records are:
pd.read_sql_query(sql=generated_sql,
                  con=connection)

...doesn't look right. What could be the issue?

In [ ]:
# Let's overwrite the prompt with updated rules:
def assemble_prompt(schema: str, question: str) -> str:
  prompt= f"""
  Write a SQLite query that answers the question below.

  Question:
  {question}

  Schema:
  {schema}

  Rules:
  - Use only the tables and columns in the schema provided. Do not invent columns or table names.
  - Generate a single read-only SELECT query.
  - 'content_rating' is an age classification, not an audience score.
  - 'duration_minutes' applies to movies; seasons applies to TV shows.
  - 'genre' contains plural records (i.e. 'Documentary' is 'Documentaries', 'Crime TV Show' is 'Crime TV Shows')
  - Return only SQL. Do not include markdown formatting or explanations.
  """
  return prompt

In [ ]:
# Let's reassemble the prompt:
prompt = assemble_prompt(schema=schema, question=question)

In [ ]:
response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt)

generated_sql = response.text.strip()

print(generated_sql)

In [ ]:
# Let's see if these records change:
pd.read_sql_query(sql=generated_sql,
                  con=connection)